# Drishti · Indian plate OCR fine-tune on Kaggle

Fine-tunes the global CCT plate OCR (fast-plate-ocr, MIT) on Indian plates and exports ONNX for Drishti.

**Before running**
1. *Settings → Accelerator:* **GPU T4** · *Internet:* **On**
2. *Add Input → Datasets:*
   - **`drishti-anpr`**: upload the Drishti project zip as a private dataset - required
     (synthetic Indian plate generator, format grammar, evaluation)
   - **`drishti-ocr-real`** (optional, strongly recommended): the output folder of
     `drishti label ... --out data/ocr_real` after review (`crops/` + the `annotations.csv` downloaded from
     `review.html`), zipped and uploaded as a private dataset.

**What it does:** renders synthetic Indian plates in parallel, splits the real crops by plate number so no
vehicle is in both train and validation, scores the current OCR, fine-tunes from the public global weights,
scores again on the same real validation plates, and zips the ONNX model to `/kaggle/working/drishti_ocr.zip`.

Without real crops the run still works, but validation then uses synthetic plates, which says little about
real roads.


In [ ]:
import os, sys
print("DRISHTI OCR v2  (40 epochs, real + 3000 hard synthetic validation plates)")
SMOKE = os.environ.get("SMOKE") == "1"

N_SYNTH = 800 if SMOKE else 120000   # synthetic Indian plates
N_SYNTH_VAL = 200 if SMOKE else 3000 # hard synthetic validation plates (all layouts, toll angle, night IR)
EPOCHS = 1 if SMOKE else 40          # ~9.5 min each on Kaggle: fits the 12 h limit with room to spare
BATCH = 32 if SMOKE else 256
LR = 5e-4
REAL_OVERSAMPLE = 6                  # each real crop counts this many times against the synthetic ones
VAL_FRACTION = 0.15
MODEL_CFG = "cct_s_v2"               # cct_xs_v2 = faster for edge devices
BACKEND = os.environ.get("KERAS_BACKEND", "tensorflow")   # Kaggle images ship TensorFlow with GPU
SEED = 0
INPUT = os.environ.get("KAGGLE_INPUT", "/kaggle/input")
WORK = os.environ.get("KAGGLE_WORK", "/kaggle/working")
SCRATCH = os.environ.get("KAGGLE_SCRATCH", "/tmp/drishti")  # big intermediate data stays out of the saved output


In [ ]:
import shutil, subprocess
from pathlib import Path

def find_repo(root):
    for pp in Path(root).rglob("pyproject.toml"):
        if 'name = "drishti-anpr"' in pp.read_text(errors="ignore"):
            return pp.parent
    raise SystemExit("Add the drishti-anpr dataset (the Drishti project upload) as an input")

src = find_repo(INPUT)
REPO = Path(WORK) / "drishti"
if src.resolve() != REPO.resolve():
    shutil.copytree(src, REPO, dirs_exist_ok=True, ignore=shutil.ignore_patterns("*.onnx", "*.keras", "__pycache__"))
os.environ["KERAS_BACKEND"] = BACKEND
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "fast-plate-ocr[train]>=1.1", "albumentations"], check=True)
subprocess.run(["bash", str(REPO / "training/synth/fetch_fonts.sh")], check=True)
sys.path.insert(0, str(REPO))
print("repo:", REPO, " keras backend:", BACKEND)


In [ ]:
import csv, os
render = REPO / "training/synth/render_plates.py"
K = max(1, min(os.cpu_count() or 1, 8))
SYN = Path(SCRATCH) / "synth"
procs = []
for k in range(K):
    n = N_SYNTH // K + (1 if k < N_SYNTH % K else 0)
    procs.append(subprocess.Popen([sys.executable, str(render), "--out", str(SYN / f"s{k}"), "--n", str(n), "--val", "0",
                                   "--seed", str(1000 + k)], stdout=subprocess.DEVNULL))
# separate seed: validation plates never appear in training
procs.append(subprocess.Popen([sys.executable, str(render), "--out", str(SYN / "val"), "--n", "0", "--val", str(N_SYNTH_VAL),
                               "--seed", "999"], stdout=subprocess.DEVNULL))
for p in procs:
    assert p.wait() == 0, "synthetic generation failed"
print(f"rendered {N_SYNTH} synthetic training plates and {N_SYNTH_VAL} validation plates with {K} workers")


In [ ]:
import random
OCR = Path(WORK) / "ocr"
OCR.mkdir(parents=True, exist_ok=True)

def rows_from(csv_path):
    base = Path(csv_path).resolve().parent
    with open(csv_path, newline="", encoding="utf-8") as f:
        return [((base / r["image_path"]).resolve(), r["plate_text"].strip().upper()) for r in csv.DictReader(f) if r.get("plate_text")]

def write_csv(path, rows):
    # fast-plate-ocr joins paths to the CSV folder as text, so keep them relative to it
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(["image_path", "plate_text"])
        for img, txt in rows:
            w.writerow([os.path.relpath(img, path.parent), txt])

synth = [r for k in range(K) for r in rows_from(SYN / f"s{k}/train/annotations.csv")]

real = []
for ann in Path(INPUT).rglob("annotations*.csv"):
    if src in ann.parents or REPO in ann.parents:
        continue  # the project's own files, not data
    rs = [r for r in rows_from(ann) if r[0].exists()]
    if rs:
        print(f"real crops: {len(rs)} from {ann}")
        real += rs
real = list({str(p): (p, t) for p, t in real}.values())  # annotations.csv and annotations_auto.csv overlap

synth_val = rows_from(SYN / "val/val/annotations.csv")
rng = random.Random(SEED)
real_val = []
if real:
    plates = sorted({t for _, t in real}); rng.shuffle(plates)
    val_plates = set(plates[: max(1, int(VAL_FRACTION * len(plates)))])
    real_val = [r for r in real if r[1] in val_plates]
    real_train = [r for r in real if r[1] not in val_plates]
    train = synth + real_train * REAL_OVERSAMPLE
    print(f"train: {len(synth)} synthetic + {len(real_train)} real (x{REAL_OVERSAMPLE})")
else:
    train = synth
    print("WARNING: no real crops found")
# Model selection uses real + hard synthetic plates together. Real crops labelled automatically are the
# easy ones (the current OCR already reads them), so on their own they sit at 100% and cannot pick a model.
val = real_val + synth_val
print(f"validation: {len(real_val)} real + {len(synth_val)} hard synthetic plates")
rng.shuffle(train)
write_csv(OCR / "train.csv", train)
write_csv(OCR / "val.csv", val)
write_csv(OCR / "val_real.csv", real_val)
write_csv(OCR / "val_synth.csv", synth_val)


In [ ]:
import numpy as np, cv2
from drishti.ocr import SlotOCR
from drishti.formats import FormatGrammar

def eval_ocr(model, config=None, csv_path=OCR / "val.csv", batch=128):
    ocr = SlotOCR(model, config)
    g = FormatGrammar.from_pack("india", ocr.alphabet, ocr.pad_char)
    rows = rows_from(csv_path)
    free = strict = 0; char_ok = char_n = 0
    for i in range(0, len(rows), batch):
        chunk = rows[i:i + batch]
        probs, _ = ocr([cv2.imread(str(p)) for p, _ in chunk])
        for (p, truth), pr in zip(chunk, probs):
            d = g.decode(np.log(pr + 1e-9), mode="auto")
            free += d.raw_text == truth; strict += d.text == truth
            char_ok += sum(a == b for a, b in zip(d.text, truth)); char_n += max(len(truth), len(d.text))
    n = max(len(rows), 1)
    return {"plates": len(rows), "exact_free": round(free / n, 4), "exact_with_grammar": round(strict / n, 4),
            "char_accuracy": round(char_ok / max(char_n, 1), 4)}

def eval_all(model, config=None):
    out = {"synthetic_hard": eval_ocr(model, config, OCR / "val_synth.csv")}
    if real_val:
        out["real"] = eval_ocr(model, config, OCR / "val_real.csv")
    return out

baseline = eval_all("cct-s-v2-global-model")
print("current global OCR:", baseline)


In [ ]:
from drishti import hub
weights = hub.fetch(f"{MODEL_CFG.replace('_', '-')}-global-model", "keras")
subprocess.run([sys.executable, str(REPO / "training/ocr/make_augmentation.py"), str(OCR / "augment.yaml")], check=True)
RUNS = Path(WORK) / "ocr_runs"
cmd = ["fast-plate-ocr", "train",
       "--model-config-file", str(REPO / f"training/ocr/{MODEL_CFG}.yaml"),
       "--plate-config-file", str(REPO / "training/ocr/india_plate_config.yaml"),
       "--annotations", str(OCR / "train.csv"), "--val-annotations", str(OCR / "val.csv"),
       "--augmentation-path", str(OCR / "augment.yaml"), "--weights-path", str(weights),
       "--lr", str(LR), "--warmup-fraction", "0.05", "--label-smoothing", "0.05",
       "--batch-size", str(BATCH), "--epochs", str(EPOCHS), "--workers", "4",
       "--early-stopping-patience", "10", "--early-stopping-metric", "val_plate_acc",
       "--output-dir", str(RUNS)]
print(" ".join(cmd))
subprocess.run(cmd, check=True, env={**os.environ, "KERAS_BACKEND": BACKEND})
run_dir = sorted(p for p in RUNS.iterdir() if (p / "best.keras").exists())[-1]
print("run:", run_dir)


In [ ]:
import json
EXP = Path(WORK) / "drishti_ocr"
if EXP.exists():
    shutil.rmtree(EXP)
EXP.mkdir(parents=True)
name = f"ocr_{MODEL_CFG}_in"
shutil.copy(run_dir / "best.keras", EXP / f"{name}.keras")  # kept even if the export below fails
shutil.copy(REPO / "training/ocr/india_plate_config.yaml", EXP / f"{name}_plate_config.yaml")
exp = subprocess.run(["fast-plate-ocr", "export", "--model", str(run_dir / "best.keras"), "--format", "onnx",
                      "--plate-config-file", str(REPO / "training/ocr/india_plate_config.yaml"), "--save-dir", str(EXP),
                      "--dynamic-batch"], env={**os.environ, "KERAS_BACKEND": BACKEND})
onnx_out = next(iter(sorted(EXP.glob("*.onnx"))), None)
if exp.returncode == 0 and onnx_out is not None:
    onnx_out.rename(EXP / f"{name}.onnx")
    finetuned = eval_all(str(EXP / f"{name}.onnx"), str(EXP / f"{name}_plate_config.yaml"))
else:
    finetuned = "ONNX export failed; the trained .keras model is in the zip and can be exported later"
    print("WARNING:", finetuned)
report = {"validation": "real" if real else "synthetic", "before": baseline, "after": finetuned}
print(json.dumps(report, indent=2))
(EXP / "metrics.json").write_text(json.dumps(report, indent=2))
for f in ("training_log.csv", "hyper_params.json"):
    if (run_dir / f).exists():
        shutil.copy(run_dir / f, EXP / f)
(EXP / "drishti.yaml").write_text(
    "# put both files in models/ and run:  drishti --config drishti.yaml image photo.jpg\n"
    f"models:\n  ocr: models/{name}.onnx\n  ocr_config: models/{name}_plate_config.yaml\n")
shutil.make_archive(str(Path(WORK) / "drishti_ocr"), "zip", EXP)
print("download:", Path(WORK) / "drishti_ocr.zip")
